# Qwen3-4B Unsloth Fine-Tuning

[`gururaser/ithaki-bilimkurgu-klasikleri`](https://huggingface.co/datasets/gururaser/ithaki-bilimkurgu-klasikleri) (Hugging Face, `cc-by-nc-4.0`) veri setinden üretilen ChatML soru-cevap veri seti ile `unsloth/Qwen3-4B-Instruct-2507` modeli, Colab **T4 GPU** üzerinde Unsloth + LoRA kullanılarak fine-tune edilir. Sonunda model Hugging Face Hub'a yüklenir ve fine-tuning öncesi/sonrası cevaplar karşılaştırılır.

> Çalıştırmadan önce: **Çalışma Zamanı > Çalışma zamanı türünü değiştir > T4 GPU** seçili olduğundan emin olun.

## 1. Kurulum

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Veri seti oluşturma

[`gururaser/ithaki-bilimkurgu-klasikleri`](https://huggingface.co/datasets/gururaser/ithaki-bilimkurgu-klasikleri) veri setindeki dolu sütunlardan (`yazar`, `cevirmen`, `yayinevi`, `kategori`, `orijinal_adi`, `sayfa_sayisi`, `satis_fiyati`, `yayin_tarihi`, `ozet`) şablon tabanlı soru-cevap çiftleri üretilir. `cevirmen` ve `orijinal_adi` gibi seyrek dolu sütunlar için soru, sadece değer mevcutsa üretilir.

In [ ]:
CSV_PATH = "hf://datasets/gururaser/ithaki-bilimkurgu-klasikleri/ithaki_bilimkurgu_klasikleri_ozetli.csv"

print(f"Kullanılacak dosya: {CSV_PATH}")

In [3]:
import pandas as pd
import random

random.seed(3407)

df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")

QUESTION_TEMPLATES = {
    "yazar": "{kitap_adi} kitabının yazarı kimdir?",
    "cevirmen": "{kitap_adi} kitabının çevirmeni kimdir?",
    "yayinevi": "{kitap_adi} kitabı hangi yayınevi tarafından yayımlanmıştır?",
    "kategori": "{kitap_adi} kitabı hangi kategoridedir?",
    "orijinal_adi": "{kitap_adi} kitabının orijinal adı nedir?",
    "sayfa_sayisi": "{kitap_adi} kitabı kaç sayfadır?",
    "satis_fiyati": "{kitap_adi} kitabının satış fiyatı nedir?",
    "yayin_tarihi": "{kitap_adi} kitabının yayın tarihi nedir?",
}

# İsim alanları (yazar/çevirmen/orijinal_adi) Türkçe ünlü uyumuna göre çekimlenir.
# CSV'deki isimler tutarsız şekilde İ/I karışık BÜYÜK harfle geliyor (örn. hem
# 'ISAAC ASIMOV' hem 'ISAAC ASİMOV' görülüyor); Python'ın .title() metodu Türkçe
# İ harfinde bozuk birleşik karakter ürettiği için (İ -> 'i̇') özel dönüşüm gerekir.
TR_INCE = set("eiöü")
TR_KALIN = set("aıou")
TR_SERT_SESSIZ = set("çfhkpsşt")

def turkce_baslik(deger: str) -> str:
    kelimeler = deger.replace("İ", "i").lower().split(" ")
    return " ".join(k[:1].upper() + k[1:] for k in kelimeler if k)

def tr_bildirme_eki(deger: str) -> str:
    harfler = [h for h in deger.lower() if h.isalpha()]
    son_unlu = next((h for h in reversed(harfler) if h in TR_INCE | TR_KALIN), "i")
    ince = son_unlu in TR_INCE
    yuvarlak = son_unlu in "oöuü"
    unlu = "ü" if ince and yuvarlak else "i" if ince else "u" if yuvarlak else "ı"
    unsuz = "t" if harfler and harfler[-1] in TR_SERT_SESSIZ else "d"
    return f"{unsuz}{unlu}r"

def isim_cevap(kitap_adi: str, alan_adi: str, deger: str) -> str:
    deger = turkce_baslik(deger)
    return f"{kitap_adi} kitabının {alan_adi} {deger}'{tr_bildirme_eki(deger)}."

ANSWER_BUILDERS = {
    "yazar": lambda k, d: isim_cevap(k, "yazarı", d),
    "cevirmen": lambda k, d: isim_cevap(k, "çevirmeni", d),
    "orijinal_adi": lambda k, d: isim_cevap(k, "orijinal adı", d),
    "yayinevi": lambda k, d: f"{k} kitabı {d} tarafından yayımlanmıştır.",
    "kategori": lambda k, d: f"{k} kitabı {d} kategorisindedir.",
    "sayfa_sayisi": lambda k, d: f"{k} kitabı {d} sayfadır.",
    "satis_fiyati": lambda k, d: f"{k} kitabının satış fiyatı {d}.",
    "yayin_tarihi": lambda k, d: f"{k} kitabının yayın tarihi {d}.",
}

OZET_SORULARI = [
    "{kitap_adi} kitabının konusu nedir?",
    "{kitap_adi} kitabı ne anlatıyor?",
]

qa_records = []
for _, row in df.iterrows():
    kitap_adi = str(row["kitap_adi"]).strip()

    for alan, soru_kalibi in QUESTION_TEMPLATES.items():
        deger = str(row[alan]).strip()
        if not deger or deger.lower() == "nan":
            continue
        qa_records.append({
            "kitap_adi": kitap_adi,
            "soru": soru_kalibi.format(kitap_adi=kitap_adi),
            "cevap": ANSWER_BUILDERS[alan](kitap_adi, deger),
        })

    ozet = str(row["ozet"]).strip()
    if ozet and ozet.lower() != "nan":
        for soru_kalibi in OZET_SORULARI:
            qa_records.append({
                "kitap_adi": kitap_adi,
                "soru": soru_kalibi.format(kitap_adi=kitap_adi),
                "cevap": ozet,
            })

print(f"Toplam QA çifti: {len(qa_records)}  |  Kitap sayısı: {df['kitap_adi'].nunique()}")
qa_records[:3]

Toplam QA çifti: 891  |  Kitap sayısı: 103


[{'kitap_adi': 'Nemesis',
  'soru': 'Nemesis kitabının yazarı kimdir?',
  'cevap': "Nemesis kitabının yazarı Isaac Asimov'dur."},
 {'kitap_adi': 'Nemesis',
  'soru': 'Nemesis kitabının çevirmeni kimdir?',
  'cevap': "Nemesis kitabının çevirmeni Münevver Uzun'dur."},
 {'kitap_adi': 'Nemesis',
  'soru': 'Nemesis kitabı hangi yayınevi tarafından yayımlanmıştır?',
  'cevap': 'Nemesis kitabı İthaki Yayınları tarafından yayımlanmıştır.'}]

In [4]:
from datasets import Dataset

# Sızıntıyı önlemek için bölme satır bazında değil KİTAP bazında yapılır:
# aynı kitabın QA çiftleri hem train hem test'e dağılmaz.
kitaplar = sorted(df["kitap_adi"].unique().tolist())
random.shuffle(kitaplar)

test_kitap_sayisi = max(1, round(len(kitaplar) * 0.1))
test_kitaplar = set(kitaplar[:test_kitap_sayisi])
train_kitaplar = set(kitaplar[test_kitap_sayisi:])

train_records = [r for r in qa_records if r["kitap_adi"] in train_kitaplar]
test_records = [r for r in qa_records if r["kitap_adi"] in test_kitaplar]

def to_conversations(records):
    return [{"conversations": [
        {"role": "user", "content": r["soru"]},
        {"role": "assistant", "content": r["cevap"]},
    ]} for r in records]

train_dataset = Dataset.from_list(to_conversations(train_records))
test_dataset = Dataset.from_list(to_conversations(test_records))

print(f"Train: {len(train_dataset)} örnek ({len(train_kitaplar)} kitap)")
print(f"Test:  {len(test_dataset)} örnek ({len(test_kitaplar)} kitap)")

Train: 804 örnek (93 kitap)
Test:  87 örnek (10 kitap)


## 3. Model yükleme (T4 için 4-bit)

In [5]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length = 1024,   # QA çiftleri kısa (özetler ~150-300 karakter) -> T4'te bellek tasarrufu
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.3: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [6]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

## 4. Fine-tuning ÖNCESİ test

Test setinden rastgele seçilen sorular, eğitim öncesi base model ile cevaplanır. Bu cevaplar eğitim sonrasıyla karşılaştırılacak.

In [7]:
random.seed(7)
ORNEK_SORULAR = random.sample(test_records, min(6, len(test_records)))

def cevap_uret(model, soru, max_new_tokens=200):
    mesajlar = [{"role": "user", "content": soru}]
    text = tokenizer.apply_chat_template(
        mesajlar, tokenize = False, add_generation_prompt = True,
    )
    inputs = tokenizer(text, return_tensors = "pt").to("cuda")
    cikti = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 0.7, top_p = 0.8, top_k = 20,
    )
    yanit = tokenizer.decode(cikti[0][inputs["input_ids"].shape[1]:], skip_special_tokens = True)
    return yanit.strip()

FastLanguageModel.for_inference(model)
oncesi_cevaplar = []
for ornek in ORNEK_SORULAR:
    yanit = cevap_uret(model, ornek["soru"])
    oncesi_cevaplar.append(yanit)
    print(f"S: {ornek['soru']}\nBeklenen: {ornek['cevap']}\nModel (öncesi): {yanit}\n{'-'*60}")

S: Liktorun Kılıcı kitabının konusu nedir?
Beklenen: Severian sürgün yolunda iktidar, sadakat ve geçmişin gizemleriyle yüzleşirken Urth'un daha geniş tarihine yaklaşır.
Model (öncesi): "Liktorun Kılıcı" adlı eser, **Tamerlan Kılıç** tarafından yazılan bir eserdir. Ancak, bu eserin yaygın bir kitap, dergi ya da bilinen bir metin kaynağı olarak bilinmiyor. Bu nedenle, "Liktorun Kılıcı" kitabının konusu olarak bilinen bir bilgi mevcut değildir.

Belki de yanlış yazılmış olabilir. Aşağıdaki olasılıklardan biri doğrulayabilir:

1. **"Liktorun Kılıcı"** kelimesi, **"Liktörün Kılıcı"** ya da **"Liktörün Kılıcı"** gibi bir hata olabilir.  
   - Bu durumda, muhtemelen **"Liktörün Kılıcı"** veya **"Liktörün Kılıcı"** gibi bir es
------------------------------------------------------------
S: Maymunlar Gezegeni kitabı hangi kategoridedir?
Beklenen: Maymunlar Gezegeni kitabı Bilimkurgu Klasikleri kategorisindedir.
Model (öncesi): "Maymunlar Gezegeni" kitabı, genellikle **bilim kurgu** kategorisind

## 5. LoRA adaptörü ekleme

In [8]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.7.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 6. Veri setini ChatML formatına dönüştürme

In [9]:
from unsloth.chat_templates import standardize_data_formats

train_dataset = standardize_data_formats(train_dataset)
test_dataset = standardize_data_formats(test_dataset)

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched = True)
test_dataset = test_dataset.map(formatting_prompts_func, batched = True)

print(train_dataset[0]["text"])

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/804 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=6):   0%|          | 0/87 [00:00<?, ? examples/s]

Map:   0%|          | 0/804 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

<|im_start|>user
Nemesis kitabının yazarı kimdir?<|im_end|>
<|im_start|>assistant
Nemesis kitabının yazarı Isaac Asimov'dur.<|im_end|>



## 7. Trainer kurulumu (T4'e göre ayarlanmış parametreler)

In [10]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # efektif batch = 8
        num_train_epochs = 3,              # küçük veri seti (~700 örnek) birkaç kez görülsün
        warmup_steps = 5,
        learning_rate = 2e-4,
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        eval_strategy = "steps",
        eval_steps = 20,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/804 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/87 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [11]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

Map:   0%|          | 0/804 [00:00<?, ? examples/s]

Map:   0%|          | 0/87 [00:00<?, ? examples/s]

## 8. Eğitim

In [12]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 804 | Num Epochs = 3 | Total steps = 303
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Step,Training Loss,Validation Loss
20,1.434600,1.044592
40,0.757700,0.972777
60,0.936500,0.962345
80,0.942600,0.984040
100,0.491800,0.966379
120,0.390900,1.034278
140,0.489300,1.029663
160,0.408700,1.082593
180,0.558800,1.062032
200,0.223100,1.069088


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## 9. Fine-tuning SONRASI test

Aynı örnek sorular, fine-tune edilmiş model ile tekrar cevaplanır.

In [13]:
FastLanguageModel.for_inference(model)

sonrasi_cevaplar = []
for ornek in ORNEK_SORULAR:
    yanit = cevap_uret(model, ornek["soru"])
    sonrasi_cevaplar.append(yanit)
    print(f"S: {ornek['soru']}\nBeklenen: {ornek['cevap']}\nModel (sonrası): {yanit}\n{'-'*60}")

S: Liktorun Kılıcı kitabının konusu nedir?
Beklenen: Severian sürgün yolunda iktidar, sadakat ve geçmişin gizemleriyle yüzleşirken Urth'un daha geniş tarihine yaklaşır.
Model (sonrası): Nükleer savaş sonrası dünyada bir hacker, bilgisayar saldırısıyla bağlantılı olağanüstü bir sırla birlikte evlerindeki entrikaların içine girer.
------------------------------------------------------------
S: Maymunlar Gezegeni kitabı hangi kategoridedir?
Beklenen: Maymunlar Gezegeni kitabı Bilimkurgu Klasikleri kategorisindedir.
Model (sonrası): Maymunlar Gezegeni kitabı Bilimkurgu Klasikleri kategorisindedir.
------------------------------------------------------------
S: Ben, Efsane kitabı ne anlatıyor?
Beklenen: Bir salgın sonrası vampirleşen dünyada son insan olduğuna inanan Robert Neville, hayatta kalma mücadelesi verirken kendi yerini sorgular.
Model (sonrası): Severian, yeni Autarch'ın yanında savaş ve entrikaların içinden geçerken Urth'un yönetimine ve kendi kaderine yaklaşır.
-----------------

## 10. Önce / Sonra karşılaştırması

In [14]:
karsilastirma_df = pd.DataFrame({
    "Soru": [o["soru"] for o in ORNEK_SORULAR],
    "Beklenen Cevap": [o["cevap"] for o in ORNEK_SORULAR],
    "Fine-tuning ÖNCESİ": oncesi_cevaplar,
    "Fine-tuning SONRASI": sonrasi_cevaplar,
})
pd.set_option("display.max_colwidth", None)
karsilastirma_df

,Soru,Beklenen Cevap,Fine-tuning ÖNCESİ,Fine-tuning SONRASI
0,Liktorun Kılıcı kitabının konusu nedir?,"Severian sürgün yolunda iktidar, sadakat ve geçmişin gizemleriyle yüzleşirken Urth'un daha geniş tarihine yaklaşır.","""Liktorun Kılıcı"" adlı eser, **Tamerlan Kılıç** tarafından yazılan bir eserdir. Ancak, bu eserin yaygın bir kitap, dergi ya da bilinen bir metin kaynağı olarak bilinmiyor. Bu nedenle, ""Liktorun Kılıcı"" kitabının konusu olarak bilinen bir bilgi mevcut değildir.\n\nBelki de yanlış yazılmış olabilir. Aşağıdaki olasılıklardan biri doğrulayabilir:\n\n1. **""Liktorun Kılıcı""** kelimesi, **""Liktörün Kılıcı""** ya da **""Liktörün Kılıcı""** gibi bir hata olabilir. \n - Bu durumda, muhtemelen **""Liktörün Kılıcı""** veya **""Liktörün Kılıcı""** gibi bir es","Nükleer savaş sonrası dünyada bir hacker, bilgisayar saldırısıyla bağlantılı olağanüstü bir sırla birlikte evlerindeki entrikaların içine girer."
1,Maymunlar Gezegeni kitabı hangi kategoridedir?,Maymunlar Gezegeni kitabı Bilimkurgu Klasikleri kategorisindedir.,"""Maymunlar Gezegeni"" kitabı, genellikle **bilim kurgu** kategorisinde sınıflandırılır.\n\nBu, kitabın temel olayları, karakterlerini ve dünyasını bilim kurgu temelleri üzerine inşa ettiğini gösterir. Kitap, özellikle **E. M. Forster** tarafından yazılmış bir eser değil; aslında ""Maymunlar Gezegeni"" adı, biraz karışıklık yaratabilir çünkü:\n\n- **""Maymunlar Gezegeni""** adlı bir eser, **E. M. Forster**'ın yazdığı bir kitap değildir.\n- Gerçekten bilinen ve yaygın olarak bilinen ""Maymunlar Gezegeni"" adlı eser, **İngiliz yazarı Arthur C. Clarke** tarafından yazılmış bir eserdir. Ancak bu da doğru değil",Maymunlar Gezegeni kitabı Bilimkurgu Klasikleri kategorisindedir.
2,"Ben, Efsane kitabı ne anlatıyor?","Bir salgın sonrası vampirleşen dünyada son insan olduğuna inanan Robert Neville, hayatta kalma mücadelesi verirken kendi yerini sorgular.","*Ben* adlı eser, **Efsane kitabı** değil, **""Ben""** adlı bir eser olarak bilinen **""Ben"" (The Book of Efsane)** değil, aslında **""Ben""** adlı bir eser olan **""Ben""** (İngilizce: *Me*) olabilir. Ancak bu bağlamda bir karışıklık var.\n\nAma muhtemelen kastedilen eser, **""Ben""** adlı, **Turgut Uysal** tarafından yazılan, **Efsane** ile ilgili bir eser değil, daha çok **""Ben""** adlı bir **öykü** ya da **öykü kitabı**.\n\nAncak en olası ihtimal: \nKastedilen eser **""Ben""** değil, **""Efsane""** adlı bir kitap olabilir. Ama özellikle **""Efsane""","Severian, yeni Autarch'ın yanında savaş ve entrikaların içinden geçerken Urth'un yönetimine ve kendi kaderine yaklaşır."
3,Stepford Kadınları kitabının satış fiyatı nedir?,"Stepford Kadınları kitabının satış fiyatı ₺154,00.","""Stepford Kadınları"" adlı kitabın satış fiyatı, yayıncı, ülkede ve ne zaman satın alındığında değişebilir. Bu nedenle kesin bir fiyat vermek zor olur. Ancak, kitabın orijinal olarak 1970’lerde yayımlanmış olduğu bilinir ve orijinal bir kitap olarak günümüzde genellikle **yayıncıdan satılan baskılar** veya **e-kitap formatında** bulunur.\n\nŞu anda (2024) şu bilgileri kullanarak tahmini bir fiyat veririz:\n\n- **Orijinal kitap (çeviri veya orijinal):** \n - Türkiye’deki bazı kitap mağazalarında (örneğin, İngilizce orijinali için) **100 TL – 150 TL** arasında satılır.","Stepford Kadınları kitabının satış fiyatı ₺145,00."
4,Beden Kemiricilerin İstilası kitabının satış fiyatı nedir?,"Beden Kemiricilerin İstilası kitabının satış fiyatı ₺196,00.","""Beden Kemiricilerin İstilası"" adlı kitabı, Türkiye'de genellikle **kültür ve tarih konularında** geçen bir eser olarak bilinir. Ancak bu kitap adı, özellikle de bilinen bir tarih eseri ya da bilinen bir yazarın eseri olarak yaygın bir kaynaktan değil, **daha çok bir şakacı, kurgusal ya da deneysel bir yapıya sahip bir metin** olarak bilinir.\n\nBu nedenle, bu kitabın **resmi satış fiyatı** olarak bilinen bir veri yoktur çünkü:\n\n- Kitabın yazarı, yayınevi veya tarih bilgisi doğrultusunda net bir kaynak bulunmamaktadır.\n- ""Beden Kemiricilerin

## 11. Hugging Face Hub'a yükleme

Aşağıdaki hücre bir giriş penceresi açar; Hugging Face **write** izinli bir erişim token'ı girin.

In [15]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
HF_USERNAME = "HF_USERNAME"  # kendi Hugging Face kullanıcı adınızla değiştirin
REPO_ADI = f"{HF_USERNAME}/qwen3-4b-ithaki-bilimkurgu"

model.push_to_hub_merged(REPO_ADI, tokenizer, save_method = "merged_16bit")

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...bilimkurgu/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0002-of-00002.safetensors:   0%|          | 4.22MB / 3.08GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [04:39<00:00, 139.73s/it]


Unsloth: Merge process complete. Saved to `/content/gururaser/qwen3-4b-ithaki-bilimkurgu`
